In [87]:
import math


def f(x: float, y: float) -> float:
    return math.cos(x + 2) - 0.3 * y ** 2

In [88]:
def eiler(a: float, b: float, n: int, y0: float, eps: float, y_last: list[float]|None) -> tuple[list[float], list[float], list[float], list[float]]:
    xs = [a]
    ys = [y0]
    h = (b - a) / n
    for i in range(0, n):
        yi_2 = ys[i] + h / 2 * f(xs[i], ys[i])
        ys.append((ys[i] + h * f(xs[i] + h / 2, yi_2)))
        xs.append(xs[i] + h)

    if y_last is not None:
        dy = [abs(y_last[i] - ys[2 * i]) for i in range(len(y_last))]
        if max(dy) < eps:
            return xs, [round(y, 4) for y in y_last], [round(y, 4) for y in ys], [y for y in dy]

    return eiler(a, b, n * 2, y0, eps, ys)

In [89]:
a = 0
b = 0.5
y0 = 0
n = 8
eps = 1e-3

y_eiler = eiler(a, b, n, y0, eps, None)
print(y_eiler[0], y_eiler[1], y_eiler[2], y_eiler[3], sep='\n')

[0, 0.03125, 0.0625, 0.09375, 0.125, 0.15625, 0.1875, 0.21875, 0.25, 0.28125, 0.3125, 0.34375, 0.375, 0.40625, 0.4375, 0.46875, 0.5]
[0, -0.0278, -0.059, -0.0937, -0.1316, -0.1728, -0.2172, -0.2646, -0.315]
[0, -0.0134, -0.0278, -0.043, -0.059, -0.0759, -0.0937, -0.1122, -0.1316, -0.1518, -0.1728, -0.1946, -0.2172, -0.2405, -0.2646, -0.2894, -0.315]
[0, 2.266543560839468e-06, 4.2427832590632986e-06, 5.864725783760738e-06, 7.0733313080051374e-06, 7.812844937005181e-06, 8.028844073443286e-06, 7.666003993656023e-06, 6.665564045882366e-06]


In [90]:
def runge_kutt(a: float, b: float, n: int, y0: float, eps: float, y_last: list[float]|None) -> tuple[list[float], list[float], list[float], list[float]]:
    xs = [a]
    ys = [y0]
    h = (b - a) / n

    for i in range(0, n):
        k1 = h * f(xs[i], ys[i])
        k2 = h * f(xs[i] + h/2, ys[i] + k1/2)
        k3 = h * f(xs[i] + h/2, ys[i] + k2/2)
        k4 = h * f(xs[i] + h, ys[i] + k3)

        dy = 1/6 * (k1 + 2*k2 + 2*k3 + k4)

        ys.append(ys[i] + dy)
        xs.append(xs[i] + h)

    if y_last is not None:
        dy = [abs(y_last[i] - ys[2 * i]) for i in range(len(y_last))]
        if max(dy) < eps:
            return xs, [round(y, 4) for y in y_last], [round(y, 4) for y in ys], [y for y in dy]

    return runge_kutt(a, b, n * 2, y0, eps, ys)

In [91]:
y_rk = runge_kutt(a, b, n, y0, eps, None)
print(y_rk[0], y_rk[1], y_rk[2], y_rk[3], sep='\n')

[0, 0.03125, 0.0625, 0.09375, 0.125, 0.15625, 0.1875, 0.21875, 0.25, 0.28125, 0.3125, 0.34375, 0.375, 0.40625, 0.4375, 0.46875, 0.5]
[0, -0.0278, -0.059, -0.0937, -0.1316, -0.1728, -0.2172, -0.2646, -0.315]
[0, -0.0134, -0.0278, -0.043, -0.059, -0.0759, -0.0937, -0.1122, -0.1316, -0.1518, -0.1728, -0.1946, -0.2172, -0.2405, -0.2646, -0.2894, -0.315]
[0, 3.0636711223497315e-09, 6.100519700513729e-09, 9.121736505024458e-09, 1.213962530499657e-08, 1.516719022598423e-08, 1.821781492328256e-08, 2.130504123121213e-08, 2.4442450119455117e-08]


In [92]:
def g(x: float, y: float) -> float:
    return 1 - math.sin(1.5 * x ** 2 + y)

In [93]:
def runge_kutt_system_step(a: float, h: float, step: int, y0: float, z0: float) -> tuple[list[float], list[float], list[float]]:
    xs = [a]
    ys = [y0]
    zs = [z0]

    for i in range(0, step - 1):
        k1 = h * zs[i]
        l1 = h * g(xs[i], ys[i])

        k2 = h * (zs[i] + l1/2)
        l2 = h * g(xs[i] + h/2, ys[i] + k1/2)

        k3 = h * (zs[i] + l2/2)
        l3 = h * g(xs[i] + h/2, ys[i] + k2/2)

        k4 = h * (zs[i] + l3)
        l4 = h * g(xs[i] + h, ys[i] + k3)

        dy = 1/6 * (k1 + 2*k2 + 2*k3 + k4)
        ys.append(ys[i] + dy)

        dz = 1/6 * (l1 + 2*l2 + 2*l3 + l4)
        zs.append(zs[i] + dz)
        xs.append(xs[i] + h)

    return xs, ys, zs

In [94]:
def adams3(a: float, b: float, n: int, y0: float, z0:float, eps: float, y_last: list[float]|None) -> tuple[list[float], list[float], list[float], list[float]]:
    h = (b - a) / n
    xs, ys, zs = runge_kutt_system_step(a, h, 3, y0, z0)
    for i in range(2, n):
        ys.append(ys[i] + h/12 * (23 * zs[i] - 16 * zs[i - 1] + 5 * zs[i - 2]))
        zs.append(zs[i] + h/12 * (23 * g(xs[i], ys[i]) - 16 * g(xs[i - 1], ys[i - 1]) + 5 * g(xs[i - 2], ys[i - 2])))
        xs.append(xs[i] + h)

    if y_last is not None:
        dy = [abs(y_last[i] - ys[2 * i]) for i in range(len(y_last))]
        if max(dy) < eps:
            return xs, [round(y, 4) for y in y_last], [round(y, 4) for y in ys], [y for y in dy]

    return adams3(a, b, n * 2, y0, z0, eps, ys)

In [95]:
def adams4(a: float, b: float, n: int, y0: float, z0:float, eps: float, y_last: list[float]|None) -> tuple[list[float], list[float], list[float], list[float]]:
    h = (b - a) / n
    xs, ys, zs = runge_kutt_system_step(a, h, 4, y0, z0)
    for i in range(3, n):
        ys.append(ys[i] + h/24 * (55 * zs[i] - 59 * zs[i - 1] + 37 * zs[i - 2] - 9 * zs[i - 3]))
        zs.append(zs[i] + h/24 * (55 * g(xs[i], ys[i]) - 59 * g(xs[i - 1], ys[i - 1]) + 37 * g(xs[i - 2], ys[i - 2]) - 9 * g(xs[i - 3], ys[i - 3])))
        xs.append(xs[i] + h)

    if y_last is not None:
        dy = [abs(y_last[i] - ys[2 * i]) for i in range(len(y_last))]
        if max(dy) < eps:
            return xs, [round(y, 4) for y in y_last], [round(y, 4) for y in ys], [y for y in dy]

    return adams4(a, b, n * 2, y0, z0, eps, ys)

In [96]:
z0 = 1
x, y_pre, y_l, err = adams3(a, b, n, y0, z0, eps, None)
print(x, y_pre, y_l, err, sep='\n')

[0, 0.03125, 0.0625, 0.09375, 0.125, 0.15625, 0.1875, 0.21875, 0.25, 0.28125, 0.3125, 0.34375, 0.375, 0.40625, 0.4375, 0.46875, 0.5]
[0, 0.0644, 0.1324, 0.2038, 0.2781, 0.3548, 0.4335, 0.5138, 0.5952]
[0, 0.0317, 0.0644, 0.098, 0.1324, 0.1677, 0.2038, 0.2406, 0.278, 0.3161, 0.3548, 0.3939, 0.4335, 0.4735, 0.5138, 0.5544, 0.5952]
[0, 9.757838342894232e-09, 2.6747556084727453e-06, 1.614725141524387e-05, 2.9396090993505375e-05, 3.5929639883469644e-05, 3.251136962240109e-05, 1.5730047047246565e-05, 1.7694877097462758e-05]


In [97]:
x, y_pre, y_l, err = adams4(a, b, n, y0, z0, eps, None)
print(x, y_pre, y_l, err, sep='\n')

[0, 0.03125, 0.0625, 0.09375, 0.125, 0.15625, 0.1875, 0.21875, 0.25, 0.28125, 0.3125, 0.34375, 0.375, 0.40625, 0.4375, 0.46875, 0.5]
[0, 0.0644, 0.1324, 0.2038, 0.278, 0.3547, 0.4335, 0.5138, 0.5952]
[0, 0.0317, 0.0644, 0.098, 0.1324, 0.1677, 0.2038, 0.2406, 0.278, 0.3161, 0.3548, 0.3939, 0.4335, 0.4735, 0.5138, 0.5544, 0.5952]
[0, 9.757838342894232e-09, 1.919192968435901e-08, 2.0072111672764237e-07, 1.882832600885731e-06, 7.878542004013944e-06, 1.5695910303259453e-05, 2.709907929787647e-05, 4.178034129276309e-05]
